# 06 — Concentration (geography, HHI, high-LVR, product mix & channel)

**Plain English:** Where is the risk *concentrated*? We slice exposure and
default by geography (state) and by vintage, add the **HHI** (a single
concentration number) for the state book, a **high-LVR concentration** view
(exposure by original loan-to-value band), the **product mix** (owner-occupier vs
investor, and loan purpose), and **acquisition channel** (retail vs broker /
correspondent / TPO). Concentration is a portfolio risk in its own right
(APS 220 para 35), and APRA expects higher-risk products and third-party-originated
loans to be watched specifically (APS 220 paras 35(b)/39) — these tables feed both
the appetite limits (notebook 07) and the monitoring pack.

**One-line terms**
- **Concentration** — how exposure clusters in a few states / cohorts / risky products; a portfolio risk in itself.
- **Exposure (UPB)** — current unpaid principal balance, i.e. money still at risk.
- **HHI** — Herfindahl–Hirschman Index, Σ(share %)² on a 0–10,000 scale; <1500 low · 1500–2500 moderate · >2500 high.
- **High-LVR** — loans whose *original* loan-to-value was above 90% (a higher-risk product, APS 220 para 35).
- **Product mix** — owner-occupier / investor / second-home (occupancy) and purchase / cash-out / no-cash refi (purpose); investor and cash-out are higher-risk products (APS 220 para 35(b)).
- **Channel** — how the loan was originated: Retail (lender's own desk) vs Broker / Correspondent / TPO (third party). Third-party loans get enhanced monitoring (APS 220 para 39; APG 220 paras 307–308).
- **APS 330-style** — laid out like an APRA APS 330 credit-risk disclosure table. *Format only — illustrative, not a regulatory submission.*

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")

monitor library loaded — vintages: ['2007', '2008', '2015']


In [2]:
# Latest snapshot per loan for a point-in-time concentration view --------
latest = panel.sort_values(["loan_seq", "mob"]).groupby("loan_seq").tail(1)
active = latest[latest.zb_code.isna()].copy()  # still on book

by_state = (active.groupby("prop_state")
            .agg(loans=("loan_seq", "size"),
                 exposure_upb=("cur_upb", "sum"),
                 pct_90plus=("bucket", lambda s: round(100 * s.isin(["90+", "Default"]).mean(), 2)))
            .sort_values("exposure_upb", ascending=False))
by_state["exposure_share_pct"] = (100 * by_state.exposure_upb / by_state.exposure_upb.sum()).round(2)
by_state.head(12)

,loans,exposure_upb,pct_90plus,exposure_share_pct
prop_state,,,,
CA,1727,2.995965e+08,0.46,17.26
NY,820,1.270075e+08,0.85,7.32
FL,955,1.095006e+08,0.94,6.31
TX,1017,1.032622e+08,0.69,5.95
IL,697,7.439506e+07,1.15,4.29
VA,451,6.175894e+07,0.22,3.56
NJ,428,6.135025e+07,0.47,3.53
PA,587,5.881619e+07,0.68,3.39
MA,336,5.157980e+07,0.60,2.97


In [3]:
# Concentration by vintage (APS 330-style layout) -----------------------
by_vintage = (active.groupby("vintage")
              .agg(loans=("loan_seq", "size"),
                   exposure_upb=("cur_upb", "sum"),
                   avg_credit_score=("credit_score", "mean"),
                   pct_stage2=("stage", lambda s: round(100 * (s == "Stage 2").mean(), 2)),
                   pct_stage3=("stage", lambda s: round(100 * (s == "Stage 3").mean(), 2)))
              .reset_index())
by_vintage["exposure_upb"] = by_vintage.exposure_upb.round(0)
by_vintage["avg_credit_score"] = by_vintage.avg_credit_score.round(0)
by_vintage.to_csv(m.OUT_TABLES / "06_concentration_vintage.csv", index=False)
by_state.round(2).to_csv(m.OUT_TABLES / "06_concentration_state.csv")
by_vintage

,vintage,loans,exposure_upb,avg_credit_score,pct_stage2,pct_stage3
0,2007,1378,1.140695e+08,706.0,4.14,2.03
1,2008,1163,1.097066e+08,736.0,3.35,1.46
2,2015,11976,1.511844e+09,750.0,1.29,0.47


In [4]:
# HHI of the state book + high-LVR concentration (APS 220 para 35) -------
# HHI: one number for "how concentrated is the geography?" (companion commercial
# monitor reports the same measure). Computed on exposure shares in percent.
state_share = 100 * active.groupby("prop_state")["cur_upb"].sum() / active["cur_upb"].sum()
geo_hhi = m.hhi(state_share.values)
hhi_tbl = pd.DataFrame([{
    "dimension": "state (geography)",
    "n_buckets": int(state_share.shape[0]),
    "top_share_pct": round(float(state_share.max()), 2),
    "HHI": round(geo_hhi, 0),
    "classification": m.hhi_class(geo_hhi),
}])
hhi_tbl.to_csv(m.OUT_TABLES / "06_concentration_hhi.csv", index=False)
hhi_tbl

,dimension,n_buckets,top_share_pct,HHI,classification
0,state (geography),54,17.26,568.0,Low (<1500)


In [5]:
# High-LVR concentration: exposure share by ORIGINAL LVR band ------------
active = active.copy()
active["lvr_band"] = m.lvr_band(active["ltv"])
lvr = (active.groupby("lvr_band", observed=False)
       .agg(loans=("loan_seq", "size"),
            exposure_upb=("cur_upb", "sum"),
            pct_90plus=("bucket", lambda s: round(100 * s.isin(["90+", "Default"]).mean(), 2)))
       )
lvr["exposure_share_pct"] = (100 * lvr.exposure_upb / lvr.exposure_upb.sum()).round(2)
high_lvr_share = float(lvr.loc[lvr.index.isin([">95", "90-95"]), "exposure_share_pct"].sum())
print(f"High-LVR (original LVR > 90%) share of exposure: {high_lvr_share:.2f}%")
lvr.round(2).to_csv(m.OUT_TABLES / "06_concentration_lvr.csv")
lvr.round(2)

High-LVR (original LVR > 90%) share of exposure: 12.99%


,loans,exposure_upb,pct_90plus,exposure_share_pct
lvr_band,,,,
<=60,3486,3.511384e+08,0.40,20.23
60-70,2110,2.527098e+08,0.52,14.56
70-80,5631,7.132079e+08,0.80,41.09
80-90,1483,1.929821e+08,1.08,11.12
90-95,1468,1.913157e+08,0.95,11.02
>95,338,3.422979e+07,0.30,1.97


In [6]:
# Higher-risk PRODUCT concentration: occupancy & loan purpose (APS 220 35b) --
# Investor and cash-out-refi lending are higher-risk products APRA expects to be
# limited and watched separately (worked example 5.1). The raw file carries both
# fields; here we turn them into the same concentration view as geography.
occ = m.category_concentration(active, "occupancy", m.OCCUPANCY_LABEL)
purpose = m.category_concentration(active, "loan_purpose", m.PURPOSE_LABEL)
occ.round(2).to_csv(m.OUT_TABLES / "06_concentration_occupancy.csv")
purpose.round(2).to_csv(m.OUT_TABLES / "06_concentration_purpose.csv")
inv_share = float(occ.loc[occ.index == "Investor", "exposure_share_pct"].sum())
print(f"Investor-mortgage share of exposure: {inv_share:.2f}%")
occ.round(2)

Investor-mortgage share of exposure: 8.48%


,loans,exposure_upb,pct_90plus,exposure_share_pct
occupancy,,,,
Owner-occupier,12553,1.516786e+09,0.69,87.39
Investor,1379,1.471055e+08,0.80,8.48
Second home,585,7.172854e+07,0.51,4.13


In [7]:
# Third-party / channel monitoring (APS 220 para 39; APG 220 307-308) --------
# Loans originated through brokers / correspondents / TPO are made away from the
# lender's own desk, so APRA expects their performance to be tracked separately
# from retail. Lifetime ever-90+/default by channel + current exposure share.
chan = m.channel_performance(panel, active)
chan.to_csv(m.OUT_TABLES / "06_channel_performance.csv", index=False)
tp = chan.loc[chan.third_party, "exposure_share_pct"].sum()
print(f"Third-party-originated (broker/correspondent/TPO) share of exposure: {tp:.2f}%")
chan

Third-party-originated (broker/correspondent/TPO) share of exposure: 42.63%


,channel,loans,ever_90plus_pct,active_loans,exposure_upb,exposure_share_pct,third_party
0,Retail,77857,7.92,8992,995672044.0,57.37,False
1,Correspondent,24947,5.67,3506,492281341.0,28.36,True
2,Broker,10970,8.96,1127,166512043.0,9.59,True
3,Third-party (TPO),36226,17.54,892,81154561.0,4.68,True


In [8]:
# Current / indexed-LVR concentration (Basel CRE36.140) ------------------
# Origination LVR is a static snapshot; continuous collateral monitoring needs
# CURRENT LVR, marking each property to market with the (illustrative) HPI path in
# monitor.py. Crisis-vintage survivors lever UP early; calm-vintage survivors
# de-lever as prices rise — so the current-LVR book differs from the original one.
cur_lvr = m.current_lvr_concentration(active)
cur_high = float(cur_lvr.loc[cur_lvr.index.isin([">95", "90-95"]), "exposure_share_pct"].sum())
print(f"Current/indexed high-LVR (>90%) share: {cur_high:.2f}%  "
      f"(vs {high_lvr_share:.2f}% on the original-LVR basis)")
cur_lvr.round(2).to_csv(m.OUT_TABLES / "06_concentration_current_lvr.csv")
cur_lvr.round(2)

Current/indexed high-LVR (>90%) share: 0.00%  (vs 12.99% on the original-LVR basis)


,loans,exposure_upb,pct_90plus,exposure_share_pct
current_lvr_band,,,,
<=60,14497,1.732272e+09,0.7,99.81
60-70,11,1.804205e+06,0.0,0.10
70-80,7,1.338656e+06,0.0,0.08
80-90,2,2.051650e+05,0.0,0.01
90-95,0,0.000000e+00,NaN,0.00
>95,0,0.000000e+00,NaN,0.00


In [9]:
# Single-name / large-exposure concentration (APG 220 paras 77-80) -------
# Expected to be immaterial for a granular RETAIL mortgage pool (no single borrower
# dominates) — reported so the large-exposure dimension is not silently omitted.
topn = m.topn_concentration(active)
topn.to_csv(m.OUT_TABLES / "06_concentration_topn.csv", index=False)
topn

,group,exposure_upb,share_of_book_pct
0,Top 10 loans,6358354.0,0.366
1,Top 20 loans,11898860.0,0.686
2,Top 50 loans,26712933.0,1.539
